In [ ]:
# ============================================================
# ALEXNET + 5-FOLD CV + BAYESIAN OPTIMIZATION
# FIXED TEST SET (20%) | TRAIN = 80%
# NO TRADITIONAL AUGMENTATION
# IMAGE SIZE: 256x256 | CLASSES: 3
# ============================================================

# !pip install -q keras-tuner

# ---------------- IMPORTS ----------------
import os, cv2
import numpy as np
import tensorflow as tf
import keras_tuner as kt
from tensorflow.keras import layers, models
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------- CONFIG ----------------
DATASET_DIR = "/kaggle/input/potatoleafplantvillage/VanillaLDMAugmented"
TRAIN_DIR = os.path.join(DATASET_DIR, "train")
TEST_DIR  = os.path.join(DATASET_DIR, "test")

IMG_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 10
NUM_CLASSES = 3
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

# ---------------- LOAD FUNCTION ----------------
def load_paths_labels(directory):
    image_paths, labels = [], []
    class_names = sorted(os.listdir(directory))[:NUM_CLASSES]

    for idx, cls in enumerate(class_names):
        cls_dir = os.path.join(directory, cls)
        for f in os.listdir(cls_dir):
            image_paths.append(os.path.join(cls_dir, f))
            labels.append(idx)

    return np.array(image_paths), np.array(labels), class_names

# ---------------- LOAD DATA ----------------
X_train_all, y_train_all, class_names = load_paths_labels(TRAIN_DIR)
X_test, y_test, _ = load_paths_labels(TEST_DIR)

print("Train images:", len(X_train_all))
print("Test images (Fixed):", len(X_test))

# ---------------- DATA GENERATOR ----------------
def data_generator(paths, labels):
    while True:
        idxs = np.arange(len(paths))
        np.random.shuffle(idxs)

        for i in range(0, len(paths), BATCH_SIZE):
            batch_idx = idxs[i:i+BATCH_SIZE]
            imgs, labs = [], []

            for j in batch_idx:
                img = cv2.imread(paths[j])
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                img = img.astype("float32") / 255.0

                imgs.append(img)
                labs.append(labels[j])

            imgs = np.array(imgs)
            labs = tf.keras.utils.to_categorical(labs, NUM_CLASSES)
            yield imgs, labs

# ---------------- MODEL BUILDER (BAYESIAN) ----------------
def build_model(hp):

    model = models.Sequential()

    # AlexNet-style architecture
    model.add(layers.Conv2D(96, (11,11), strides=4, activation="relu",
                            input_shape=(IMG_SIZE, IMG_SIZE, 3)))
    model.add(layers.MaxPooling2D((3,3), strides=2))

    model.add(layers.Conv2D(256, (5,5), padding="same", activation="relu"))
    model.add(layers.MaxPooling2D((3,3), strides=2))

    model.add(layers.Conv2D(384, (3,3), padding="same", activation="relu"))
    model.add(layers.Conv2D(384, (3,3), padding="same", activation="relu"))
    model.add(layers.Conv2D(256, (3,3), padding="same", activation="relu"))
    model.add(layers.MaxPooling2D((3,3), strides=2))

    model.add(layers.Flatten())

    units = hp.Int("dense_units", 512, 2048, step=512)
    model.add(layers.Dense(units, activation="relu"))

    drop = hp.Float("dropout", 0.3, 0.6, step=0.1)
    model.add(layers.Dropout(drop))

    model.add(layers.Dense(NUM_CLASSES, activation="softmax"))

    lr = hp.Choice("lr", [1e-3, 1e-4, 5e-5])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

# ---------------- 5-FOLD CV ----------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

fold = 1
cv_scores = []

for train_idx, val_idx in skf.split(X_train_all, y_train_all):

    print(f"\n🔁 Fold {fold}")

    X_train, X_val = X_train_all[train_idx], X_train_all[val_idx]
    y_train, y_val = y_train_all[train_idx], y_train_all[val_idx]

    tuner = kt.BayesianOptimization(
        build_model,
        objective="val_accuracy",
        max_trials=5,
        directory="bayesian_tuning_alexnet",
        project_name=f"fold_{fold}"
    )

    tuner.search(
        data_generator(X_train, y_train),
        steps_per_epoch=len(X_train)//BATCH_SIZE,
        validation_data=data_generator(X_val, y_val),
        validation_steps=len(X_val)//BATCH_SIZE,
        epochs=EPOCHS,
        verbose=0
    )

    best_model = tuner.get_best_models(1)[0]

    val_loss, val_acc = best_model.evaluate(
        data_generator(X_val, y_val),
        steps=len(X_val)//BATCH_SIZE,
        verbose=0
    )

    print(f"Fold {fold} Validation Accuracy: {val_acc:.4f}")
    cv_scores.append(val_acc)
    fold += 1

print("\n✅ Mean CV Accuracy:", np.mean(cv_scores))

# ---------------- FINAL MODEL ----------------
final_model = tuner.get_best_models(1)[0]

# ---------------- TEST EVALUATION (FIXED) ----------------
y_true, y_pred = [], []
test_gen = data_generator(X_test, y_test)

for _ in range(len(X_test)//BATCH_SIZE):
    imgs, labs = next(test_gen)
    preds = final_model.predict(imgs, verbose=0)
    y_true.extend(np.argmax(labs, axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

test_acc = np.mean(np.array(y_true) == np.array(y_pred))
print(f"\n🎯 Final Test Accuracy (Fixed): {test_acc:.4f}")

# ---------------- REPORT ----------------
print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# ---------------- CONFUSION MATRIX ----------------
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=class_names,
            yticklabels=class_names,
            cmap="Blues")
plt.title("Confusion Matrix (Fixed Test Set)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
# ============================================================
# VGG19 + 5-FOLD CV + BAYESIAN OPTIMIZATION
# FIXED TEST SET (20%) | TRAIN = 80%
# NO TRADITIONAL AUGMENTATION
# IMAGE SIZE: 256x256 | CLASSES: 3
# ============================================================

# !pip install -q keras-tuner

# ---------------- IMPORTS ----------------
import os, cv2
import numpy as np
import tensorflow as tf
import keras_tuner as kt
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG19
from tensorflow.keras.applications.vgg19 import preprocess_input
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------- CONFIG ----------------
DATASET_DIR = "/kaggle/input/potatoleafplantvillage/VanillaLDMAugmented"
TRAIN_DIR = os.path.join(DATASET_DIR, "train")
TEST_DIR  = os.path.join(DATASET_DIR, "test")

IMG_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 10
NUM_CLASSES = 3
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

# ---------------- LOAD FUNCTION ----------------
def load_paths_labels(directory):
    image_paths, labels = [], []
    class_names = sorted(os.listdir(directory))[:NUM_CLASSES]

    for idx, cls in enumerate(class_names):
        cls_dir = os.path.join(directory, cls)
        for f in os.listdir(cls_dir):
            image_paths.append(os.path.join(cls_dir, f))
            labels.append(idx)

    return np.array(image_paths), np.array(labels), class_names

# ---------------- LOAD DATA ----------------
X_train_all, y_train_all, class_names = load_paths_labels(TRAIN_DIR)
X_test, y_test, _ = load_paths_labels(TEST_DIR)

print("Train images:", len(X_train_all))
print("Test images (Fixed):", len(X_test))

# ---------------- DATA GENERATOR ----------------
def data_generator(paths, labels):
    while True:
        idxs = np.arange(len(paths))
        np.random.shuffle(idxs)

        for i in range(0, len(paths), BATCH_SIZE):
            batch_idx = idxs[i:i+BATCH_SIZE]
            imgs, labs = [], []

            for j in batch_idx:
                img = cv2.imread(paths[j])
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                img = img.astype("float32")
                img = preprocess_input(img)

                imgs.append(img)
                labs.append(labels[j])

            imgs = np.array(imgs)
            labs = tf.keras.utils.to_categorical(labs, NUM_CLASSES)
            yield imgs, labs

# ---------------- MODEL BUILDER (BAYESIAN) ----------------
def build_model(hp):
    base = VGG19(
        weights="imagenet",
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = False

    x = layers.Flatten()(base.output)

    units = hp.Int("dense_units", 512, 2048, step=512)
    x = layers.Dense(units, activation="relu")(x)

    drop = hp.Float("dropout", 0.3, 0.6, step=0.1)
    x = layers.Dropout(drop)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(base.input, outputs)

    lr = hp.Choice("lr", [1e-3, 1e-4, 5e-5])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# ---------------- 5-FOLD CV ----------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

fold = 1
cv_scores = []

for train_idx, val_idx in skf.split(X_train_all, y_train_all):

    print(f"\n🔁 Fold {fold}")

    X_train, X_val = X_train_all[train_idx], X_train_all[val_idx]
    y_train, y_val = y_train_all[train_idx], y_train_all[val_idx]

    tuner = kt.BayesianOptimization(
        build_model,
        objective="val_accuracy",
        max_trials=5,
        directory="bayesian_tuning_vgg19_noaug",
        project_name=f"fold_{fold}"
    )

    tuner.search(
        data_generator(X_train, y_train),
        steps_per_epoch=len(X_train)//BATCH_SIZE,
        validation_data=data_generator(X_val, y_val),
        validation_steps=len(X_val)//BATCH_SIZE,
        epochs=EPOCHS,
        verbose=0
    )

    best_model = tuner.get_best_models(1)[0]

    val_loss, val_acc = best_model.evaluate(
        data_generator(X_val, y_val),
        steps=len(X_val)//BATCH_SIZE,
        verbose=0
    )

    print(f"Fold {fold} Validation Accuracy: {val_acc:.4f}")
    cv_scores.append(val_acc)
    fold += 1

print("\n✅ Mean CV Accuracy:", np.mean(cv_scores))

# ---------------- FINAL MODEL ----------------
final_model = tuner.get_best_models(1)[0]

# ---------------- TEST EVALUATION (FIXED) ----------------
y_true, y_pred = [], []
test_gen = data_generator(X_test, y_test)

for _ in range(len(X_test)//BATCH_SIZE):
    imgs, labs = next(test_gen)
    preds = final_model.predict(imgs, verbose=0)
    y_true.extend(np.argmax(labs, axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

test_acc = np.mean(np.array(y_true) == np.array(y_pred))
print(f"\n🎯 Final Test Accuracy (Fixed): {test_acc:.4f}")

# ---------------- REPORT ----------------
print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# ---------------- CONFUSION MATRIX ----------------
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=class_names,
            yticklabels=class_names,
            cmap="Blues")
plt.title("Confusion Matrix (Fixed Test Set)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
# ============================================================
# XCEPTION + 5-FOLD CV + BAYESIAN OPTIMIZATION
# FIXED TEST SET (20%) | TRAIN = 80%
# NO TRADITIONAL AUGMENTATION
# IMAGE SIZE: 256x256 | CLASSES: 3
# ============================================================

# !pip install -q keras-tuner

# ---------------- IMPORTS ----------------
import os, cv2
import numpy as np
import tensorflow as tf
import keras_tuner as kt
from tensorflow.keras import layers, models
from tensorflow.keras.applications import Xception
from tensorflow.keras.applications.xception import preprocess_input
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------- CONFIG ----------------
DATASET_DIR = "/kaggle/input/potatoleafplantvillage/VanillaLDMAugmented"
TRAIN_DIR = os.path.join(DATASET_DIR, "train")
TEST_DIR  = os.path.join(DATASET_DIR, "test")

IMG_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 10
NUM_CLASSES = 3
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

# ---------------- LOAD FUNCTION ----------------
def load_paths_labels(directory):
    image_paths, labels = [], []
    class_names = sorted(os.listdir(directory))[:NUM_CLASSES]

    for idx, cls in enumerate(class_names):
        cls_dir = os.path.join(directory, cls)
        for f in os.listdir(cls_dir):
            image_paths.append(os.path.join(cls_dir, f))
            labels.append(idx)

    return np.array(image_paths), np.array(labels), class_names

# ---------------- LOAD DATA ----------------
X_train_all, y_train_all, class_names = load_paths_labels(TRAIN_DIR)
X_test, y_test, _ = load_paths_labels(TEST_DIR)

print("Train images:", len(X_train_all))
print("Test images (Fixed):", len(X_test))

# ---------------- DATA GENERATOR ----------------
def data_generator(paths, labels):
    while True:
        idxs = np.arange(len(paths))
        np.random.shuffle(idxs)

        for i in range(0, len(paths), BATCH_SIZE):
            batch_idx = idxs[i:i+BATCH_SIZE]
            imgs, labs = [], []

            for j in batch_idx:
                img = cv2.imread(paths[j])
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                img = img.astype("float32")
                img = preprocess_input(img)

                imgs.append(img)
                labs.append(labels[j])

            imgs = np.array(imgs)
            labs = tf.keras.utils.to_categorical(labs, NUM_CLASSES)
            yield imgs, labs

# ---------------- MODEL BUILDER (BAYESIAN) ----------------
def build_model(hp):
    base = Xception(
        weights="imagenet",
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = False

    x = base.output
    x = layers.GlobalAveragePooling2D()(x)

    units = hp.Int("dense_units", 512, 2048, step=512)
    x = layers.Dense(units, activation="relu")(x)

    drop = hp.Float("dropout", 0.3, 0.6, step=0.1)
    x = layers.Dropout(drop)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(base.input, outputs)

    lr = hp.Choice("lr", [1e-3, 1e-4, 5e-5])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# ---------------- 5-FOLD CV ----------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

fold = 1
cv_scores = []

for train_idx, val_idx in skf.split(X_train_all, y_train_all):

    print(f"\n🔁 Fold {fold}")

    X_train, X_val = X_train_all[train_idx], X_train_all[val_idx]
    y_train, y_val = y_train_all[train_idx], y_train_all[val_idx]

    tuner = kt.BayesianOptimization(
        build_model,
        objective="val_accuracy",
        max_trials=5,
        directory="bayesian_tuning_xception_noaug",
        project_name=f"fold_{fold}"
    )

    tuner.search(
        data_generator(X_train, y_train),
        steps_per_epoch=len(X_train)//BATCH_SIZE,
        validation_data=data_generator(X_val, y_val),
        validation_steps=len(X_val)//BATCH_SIZE,
        epochs=EPOCHS,
        verbose=0
    )

    best_model = tuner.get_best_models(1)[0]

    val_loss, val_acc = best_model.evaluate(
        data_generator(X_val, y_val),
        steps=len(X_val)//BATCH_SIZE,
        verbose=0
    )

    print(f"Fold {fold} Validation Accuracy: {val_acc:.4f}")
    cv_scores.append(val_acc)
    fold += 1

print("\n✅ Mean CV Accuracy:", np.mean(cv_scores))

# ---------------- FINAL MODEL ----------------
final_model = tuner.get_best_models(1)[0]

# ---------------- TEST EVALUATION (FIXED) ----------------
y_true, y_pred = [], []
test_gen = data_generator(X_test, y_test)

for _ in range(len(X_test)//BATCH_SIZE):
    imgs, labs = next(test_gen)
    preds = final_model.predict(imgs, verbose=0)
    y_true.extend(np.argmax(labs, axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

test_acc = np.mean(np.array(y_true) == np.array(y_pred))
print(f"\n🎯 Final Test Accuracy (Fixed): {test_acc:.4f}")

# ---------------- REPORT ----------------
print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# ---------------- CONFUSION MATRIX ----------------
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=class_names,
            yticklabels=class_names,
            cmap="Blues")
plt.title("Confusion Matrix (Fixed Test Set)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
# ============================================================
# MOBILENET V1 + 5-FOLD CV + BAYESIAN OPTIMIZATION
# FIXED TEST SET (20%) | TRAIN = 80%
# NO TRADITIONAL AUGMENTATION
# IMAGE SIZE: 256x256 | CLASSES: 3
# ============================================================

# !pip install -q keras-tuner

# ---------------- IMPORTS ----------------
import os, cv2
import numpy as np
import tensorflow as tf
import keras_tuner as kt
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.applications.mobilenet import preprocess_input
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------- CONFIG ----------------
DATASET_DIR = "/kaggle/input/potatoleafplantvillage/VanillaLDMAugmented"
TRAIN_DIR = os.path.join(DATASET_DIR, "train")
TEST_DIR  = os.path.join(DATASET_DIR, "test")

IMG_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 10
NUM_CLASSES = 3
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

# ---------------- LOAD FUNCTION ----------------
def load_paths_labels(directory):
    image_paths, labels = [], []
    class_names = sorted(os.listdir(directory))[:NUM_CLASSES]

    for idx, cls in enumerate(class_names):
        cls_dir = os.path.join(directory, cls)
        for f in os.listdir(cls_dir):
            image_paths.append(os.path.join(cls_dir, f))
            labels.append(idx)

    return np.array(image_paths), np.array(labels), class_names

# ---------------- LOAD DATA ----------------
X_train_all, y_train_all, class_names = load_paths_labels(TRAIN_DIR)
X_test, y_test, _ = load_paths_labels(TEST_DIR)

print("Train images:", len(X_train_all))
print("Test images (Fixed):", len(X_test))

# ---------------- DATA GENERATOR ----------------
def data_generator(paths, labels):
    while True:
        idxs = np.arange(len(paths))
        np.random.shuffle(idxs)

        for i in range(0, len(paths), BATCH_SIZE):
            batch_idx = idxs[i:i+BATCH_SIZE]
            imgs, labs = [], []

            for j in batch_idx:
                img = cv2.imread(paths[j])
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                img = img.astype("float32")
                img = preprocess_input(img)

                imgs.append(img)
                labs.append(labels[j])

            imgs = np.array(imgs)
            labs = tf.keras.utils.to_categorical(labs, NUM_CLASSES)
            yield imgs, labs

# ---------------- MODEL BUILDER (BAYESIAN) ----------------
def build_model(hp):
    base = MobileNet(
        weights="imagenet",
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        alpha=1.0
    )
    base.trainable = False

    x = base.output
    x = layers.GlobalAveragePooling2D()(x)

    units = hp.Int("dense_units", 256, 1024, step=256)
    x = layers.Dense(units, activation="relu")(x)

    drop = hp.Float("dropout", 0.3, 0.6, step=0.1)
    x = layers.Dropout(drop)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(base.input, outputs)

    lr = hp.Choice("lr", [1e-3, 1e-4, 5e-5])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# ---------------- 5-FOLD CV ----------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

fold = 1
cv_scores = []

for train_idx, val_idx in skf.split(X_train_all, y_train_all):

    print(f"\n🔁 Fold {fold}")

    X_train, X_val = X_train_all[train_idx], X_train_all[val_idx]
    y_train, y_val = y_train_all[train_idx], y_train_all[val_idx]

    tuner = kt.BayesianOptimization(
        build_model,
        objective="val_accuracy",
        max_trials=5,
        directory="bayesian_tuning_mobilenetv1_noaug",
        project_name=f"fold_{fold}"
    )

    tuner.search(
        data_generator(X_train, y_train),
        steps_per_epoch=len(X_train)//BATCH_SIZE,
        validation_data=data_generator(X_val, y_val),
        validation_steps=len(X_val)//BATCH_SIZE,
        epochs=EPOCHS,
        verbose=0
    )

    best_model = tuner.get_best_models(1)[0]

    val_loss, val_acc = best_model.evaluate(
        data_generator(X_val, y_val),
        steps=len(X_val)//BATCH_SIZE,
        verbose=0
    )

    print(f"Fold {fold} Validation Accuracy: {val_acc:.4f}")
    cv_scores.append(val_acc)
    fold += 1

print("\n✅ Mean CV Accuracy:", np.mean(cv_scores))

# ---------------- FINAL MODEL ----------------
final_model = tuner.get_best_models(1)[0]

# ---------------- TEST EVALUATION (FIXED) ----------------
y_true, y_pred = [], []
test_gen = data_generator(X_test, y_test)

for _ in range(len(X_test)//BATCH_SIZE):
    imgs, labs = next(test_gen)
    preds = final_model.predict(imgs, verbose=0)
    y_true.extend(np.argmax(labs, axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

test_acc = np.mean(np.array(y_true) == np.array(y_pred))
print(f"\n🎯 Final Test Accuracy (Fixed): {test_acc:.4f}")

# ---------------- REPORT ----------------
print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# ---------------- CONFUSION MATRIX ----------------
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=class_names,
            yticklabels=class_names,
            cmap="Blues")
plt.title("Confusion Matrix (Fixed Test Set)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
# ============================================================
# EFFICIENTNET-B0 + 5-FOLD CV + BAYESIAN OPTIMIZATION
# FIXED TEST SET (20%) | TRAIN = 80%
# NO TRADITIONAL AUGMENTATION
# IMAGE SIZE: 256x256 | CLASSES: 3
# ============================================================

# !pip install -q keras-tuner

# ---------------- IMPORTS ----------------
import os, cv2
import numpy as np
import tensorflow as tf
import keras_tuner as kt
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------- CONFIG ----------------
DATASET_DIR = "/kaggle/input/potatoleafplantvillage/VanillaLDMAugmented"
TRAIN_DIR = os.path.join(DATASET_DIR, "train")
TEST_DIR  = os.path.join(DATASET_DIR, "test")

IMG_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 10
NUM_CLASSES = 3
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

# ---------------- LOAD FUNCTION ----------------
def load_paths_labels(directory):
    image_paths, labels = [], []
    class_names = sorted(os.listdir(directory))[:NUM_CLASSES]

    for idx, cls in enumerate(class_names):
        cls_dir = os.path.join(directory, cls)
        for f in os.listdir(cls_dir):
            image_paths.append(os.path.join(cls_dir, f))
            labels.append(idx)

    return np.array(image_paths), np.array(labels), class_names

# ---------------- LOAD DATA ----------------
X_train_all, y_train_all, class_names = load_paths_labels(TRAIN_DIR)
X_test, y_test, _ = load_paths_labels(TEST_DIR)

print("Train images:", len(X_train_all))
print("Test images (Fixed):", len(X_test))

# ---------------- DATA GENERATOR ----------------
def data_generator(paths, labels):
    while True:
        idxs = np.arange(len(paths))
        np.random.shuffle(idxs)

        for i in range(0, len(paths), BATCH_SIZE):
            batch_idx = idxs[i:i+BATCH_SIZE]
            imgs, labs = [], []

            for j in batch_idx:
                img = cv2.imread(paths[j])
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                img = img.astype("float32")
                img = preprocess_input(img)

                imgs.append(img)
                labs.append(labels[j])

            imgs = np.array(imgs)
            labs = tf.keras.utils.to_categorical(labs, NUM_CLASSES)
            yield imgs, labs

# ---------------- MODEL BUILDER (BAYESIAN) ----------------
def build_model(hp):
    base = EfficientNetB0(
        weights="imagenet",
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = False

    x = base.output
    x = layers.GlobalAveragePooling2D()(x)

    units = hp.Int("dense_units", 256, 1024, step=256)
    x = layers.Dense(units, activation="relu")(x)

    drop = hp.Float("dropout", 0.3, 0.6, step=0.1)
    x = layers.Dropout(drop)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(base.input, outputs)

    lr = hp.Choice("lr", [1e-3, 1e-4, 5e-5])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# ---------------- 5-FOLD CV ----------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

fold = 1
cv_scores = []

for train_idx, val_idx in skf.split(X_train_all, y_train_all):

    print(f"\n🔁 Fold {fold}")

    X_train, X_val = X_train_all[train_idx], X_train_all[val_idx]
    y_train, y_val = y_train_all[train_idx], y_train_all[val_idx]

    tuner = kt.BayesianOptimization(
        build_model,
        objective="val_accuracy",
        max_trials=5,
        directory="bayesian_tuning_efficientnetb0_noaug",
        project_name=f"fold_{fold}"
    )

    tuner.search(
        data_generator(X_train, y_train),
        steps_per_epoch=len(X_train)//BATCH_SIZE,
        validation_data=data_generator(X_val, y_val),
        validation_steps=len(X_val)//BATCH_SIZE,
        epochs=EPOCHS,
        verbose=0
    )

    best_model = tuner.get_best_models(1)[0]

    val_loss, val_acc = best_model.evaluate(
        data_generator(X_val, y_val),
        steps=len(X_val)//BATCH_SIZE,
        verbose=0
    )

    print(f"Fold {fold} Validation Accuracy: {val_acc:.4f}")
    cv_scores.append(val_acc)
    fold += 1

print("\n✅ Mean CV Accuracy:", np.mean(cv_scores))

# ---------------- FINAL MODEL ----------------
final_model = tuner.get_best_models(1)[0]

# ---------------- TEST EVALUATION (FIXED) ----------------
y_true, y_pred = [], []
test_gen = data_generator(X_test, y_test)

for _ in range(len(X_test)//BATCH_SIZE):
    imgs, labs = next(test_gen)
    preds = final_model.predict(imgs, verbose=0)
    y_true.extend(np.argmax(labs, axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

test_acc = np.mean(np.array(y_true) == np.array(y_pred))
print(f"\n🎯 Final Test Accuracy (Fixed): {test_acc:.4f}")

# ---------------- REPORT ----------------
print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# ---------------- CONFUSION MATRIX ----------------
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=class_names,
            yticklabels=class_names,
            cmap="Blues")
plt.title("Confusion Matrix (Fixed Test Set)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
# ============================================================
# DENSENET201 + 5-FOLD CV + BAYESIAN OPTIMIZATION
# FIXED TEST SET (20%) | TRAIN = 80%
# NO TRADITIONAL AUGMENTATION
# IMAGE SIZE: 256x256 | CLASSES: 3
# ============================================================

# !pip install -q keras-tuner

# ---------------- IMPORTS ----------------
import os, cv2
import numpy as np
import tensorflow as tf
import keras_tuner as kt
from tensorflow.keras import layers, models
from tensorflow.keras.applications import DenseNet201
from tensorflow.keras.applications.densenet import preprocess_input
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------- CONFIG ----------------
DATASET_DIR = "/kaggle/input/potatoleafplantvillage/VanillaLDMAugmented"
TRAIN_DIR = os.path.join(DATASET_DIR, "train")
TEST_DIR  = os.path.join(DATASET_DIR, "test")

IMG_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 10
NUM_CLASSES = 3
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

# ---------------- LOAD FUNCTION ----------------
def load_paths_labels(directory):
    image_paths, labels = [], []
    class_names = sorted(os.listdir(directory))[:NUM_CLASSES]

    for idx, cls in enumerate(class_names):
        cls_dir = os.path.join(directory, cls)
        for f in os.listdir(cls_dir):
            image_paths.append(os.path.join(cls_dir, f))
            labels.append(idx)

    return np.array(image_paths), np.array(labels), class_names

# ---------------- LOAD DATA ----------------
X_train_all, y_train_all, class_names = load_paths_labels(TRAIN_DIR)
X_test, y_test, _ = load_paths_labels(TEST_DIR)

print("Train images:", len(X_train_all))
print("Test images (Fixed):", len(X_test))

# ---------------- DATA GENERATOR ----------------
def data_generator(paths, labels):
    while True:
        idxs = np.arange(len(paths))
        np.random.shuffle(idxs)

        for i in range(0, len(paths), BATCH_SIZE):
            batch_idx = idxs[i:i+BATCH_SIZE]
            imgs, labs = [], []

            for j in batch_idx:
                img = cv2.imread(paths[j])
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                img = img.astype("float32")
                img = preprocess_input(img)

                imgs.append(img)
                labs.append(labels[j])

            imgs = np.array(imgs)
            labs = tf.keras.utils.to_categorical(labs, NUM_CLASSES)
            yield imgs, labs

# ---------------- MODEL BUILDER (BAYESIAN) ----------------
def build_model(hp):
    base = DenseNet201(
        weights="imagenet",
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = False

    x = base.output
    x = layers.GlobalAveragePooling2D()(x)

    units = hp.Int("dense_units", 256, 1024, step=256)
    x = layers.Dense(units, activation="relu")(x)

    drop = hp.Float("dropout", 0.3, 0.6, step=0.1)
    x = layers.Dropout(drop)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(base.input, outputs)

    lr = hp.Choice("lr", [1e-3, 1e-4, 5e-5])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# ---------------- 5-FOLD CV ----------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

fold = 1
cv_scores = []

for train_idx, val_idx in skf.split(X_train_all, y_train_all):

    print(f"\n🔁 Fold {fold}")

    X_train, X_val = X_train_all[train_idx], X_train_all[val_idx]
    y_train, y_val = y_train_all[train_idx], y_train_all[val_idx]

    tuner = kt.BayesianOptimization(
        build_model,
        objective="val_accuracy",
        max_trials=5,
        directory="bayesian_tuning_densenet201_noaug",
        project_name=f"fold_{fold}"
    )

    tuner.search(
        data_generator(X_train, y_train),
        steps_per_epoch=len(X_train)//BATCH_SIZE,
        validation_data=data_generator(X_val, y_val),
        validation_steps=len(X_val)//BATCH_SIZE,
        epochs=EPOCHS,
        verbose=0
    )

    best_model = tuner.get_best_models(1)[0]

    val_loss, val_acc = best_model.evaluate(
        data_generator(X_val, y_val),
        steps=len(X_val)//BATCH_SIZE,
        verbose=0
    )

    print(f"Fold {fold} Validation Accuracy: {val_acc:.4f}")
    cv_scores.append(val_acc)
    fold += 1

print("\n✅ Mean CV Accuracy:", np.mean(cv_scores))

# ---------------- FINAL MODEL ----------------
final_model = tuner.get_best_models(1)[0]

# ---------------- TEST EVALUATION (FIXED) ----------------
y_true, y_pred = [], []
test_gen = data_generator(X_test, y_test)

for _ in range(len(X_test)//BATCH_SIZE):
    imgs, labs = next(test_gen)
    preds = final_model.predict(imgs, verbose=0)
    y_true.extend(np.argmax(labs, axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

test_acc = np.mean(np.array(y_true) == np.array(y_pred))
print(f"\n🎯 Final Test Accuracy (Fixed): {test_acc:.4f}")

# ---------------- REPORT ----------------
print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# ---------------- CONFUSION MATRIX ----------------
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=class_names,
            yticklabels=class_names,
            cmap="Blues")
plt.title("Confusion Matrix (Fixed Test Set)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
# ============================================================
# RESNET50 + 5-FOLD CV + BAYESIAN OPTIMIZATION
# FIXED TEST SET (20%) | TRAIN = 80%
# NO TRADITIONAL AUGMENTATION
# IMAGE SIZE: 256x256 | CLASSES: 3
# ============================================================

# !pip install -q keras-tuner

# ---------------- IMPORTS ----------------
import os, cv2
import numpy as np
import tensorflow as tf
import keras_tuner as kt
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------- CONFIG ----------------
DATASET_DIR = "/kaggle/input/potatoleafplantvillage/VanillaLDMAugmented"
TRAIN_DIR = os.path.join(DATASET_DIR, "train")
TEST_DIR  = os.path.join(DATASET_DIR, "test")

IMG_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 10
NUM_CLASSES = 3
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

# ---------------- LOAD FUNCTION ----------------
def load_paths_labels(directory):
    image_paths, labels = [], []
    class_names = sorted(os.listdir(directory))[:NUM_CLASSES]

    for idx, cls in enumerate(class_names):
        cls_dir = os.path.join(directory, cls)
        for f in os.listdir(cls_dir):
            image_paths.append(os.path.join(cls_dir, f))
            labels.append(idx)

    return np.array(image_paths), np.array(labels), class_names

# ---------------- LOAD DATA ----------------
X_train_all, y_train_all, class_names = load_paths_labels(TRAIN_DIR)
X_test, y_test, _ = load_paths_labels(TEST_DIR)

print("Train images:", len(X_train_all))
print("Test images (Fixed):", len(X_test))

# ---------------- DATA GENERATOR ----------------
def data_generator(paths, labels):
    while True:
        idxs = np.arange(len(paths))
        np.random.shuffle(idxs)

        for i in range(0, len(paths), BATCH_SIZE):
            batch_idx = idxs[i:i+BATCH_SIZE]
            imgs, labs = [], []

            for j in batch_idx:
                img = cv2.imread(paths[j])
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                img = img.astype("float32")
                img = preprocess_input(img)

                imgs.append(img)
                labs.append(labels[j])

            imgs = np.array(imgs)
            labs = tf.keras.utils.to_categorical(labs, NUM_CLASSES)
            yield imgs, labs

# ---------------- MODEL BUILDER (BAYESIAN) ----------------
def build_model(hp):
    base = ResNet50(
        weights="imagenet",
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = False

    x = base.output
    x = layers.GlobalAveragePooling2D()(x)

    units = hp.Int("dense_units", 256, 1024, step=256)
    x = layers.Dense(units, activation="relu")(x)

    drop = hp.Float("dropout", 0.3, 0.6, step=0.1)
    x = layers.Dropout(drop)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(base.input, outputs)

    lr = hp.Choice("lr", [1e-3, 1e-4, 5e-5])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# ---------------- 5-FOLD CV ----------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

fold = 1
cv_scores = []

for train_idx, val_idx in skf.split(X_train_all, y_train_all):

    print(f"\n🔁 Fold {fold}")

    X_train, X_val = X_train_all[train_idx], X_train_all[val_idx]
    y_train, y_val = y_train_all[train_idx], y_train_all[val_idx]

    tuner = kt.BayesianOptimization(
        build_model,
        objective="val_accuracy",
        max_trials=5,
        directory="bayesian_tuning_resnet50_noaug",
        project_name=f"fold_{fold}"
    )

    tuner.search(
        data_generator(X_train, y_train),
        steps_per_epoch=len(X_train)//BATCH_SIZE,
        validation_data=data_generator(X_val, y_val),
        validation_steps=len(X_val)//BATCH_SIZE,
        epochs=EPOCHS,
        verbose=0
    )

    best_model = tuner.get_best_models(1)[0]

    val_loss, val_acc = best_model.evaluate(
        data_generator(X_val, y_val),
        steps=len(X_val)//BATCH_SIZE,
        verbose=0
    )

    print(f"Fold {fold} Validation Accuracy: {val_acc:.4f}")
    cv_scores.append(val_acc)
    fold += 1

print("\n✅ Mean CV Accuracy:", np.mean(cv_scores))

# ---------------- FINAL MODEL ----------------
final_model = tuner.get_best_models(1)[0]

# ---------------- TEST EVALUATION (FIXED) ----------------
y_true, y_pred = [], []
test_gen = data_generator(X_test, y_test)

for _ in range(len(X_test)//BATCH_SIZE):
    imgs, labs = next(test_gen)
    preds = final_model.predict(imgs, verbose=0)
    y_true.extend(np.argmax(labs, axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

test_acc = np.mean(np.array(y_true) == np.array(y_pred))
print(f"\n🎯 Final Test Accuracy (Fixed): {test_acc:.4f}")

# ---------------- REPORT ----------------
print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# ---------------- CONFUSION MATRIX ----------------
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=class_names,
            yticklabels=class_names,
            cmap="Blues")
plt.title("Confusion Matrix (Fixed Test Set)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
# ============================================================
# INCEPTIONV3 + 5-FOLD CV + BAYESIAN OPTIMIZATION
# FIXED TEST SET (20%) | TRAIN = 80%
# NO TRADITIONAL AUGMENTATION
# IMAGE SIZE: 256x256 | CLASSES: 3
# ============================================================

# !pip install -q keras-tuner

# ---------------- IMPORTS ----------------
import os, cv2
import numpy as np
import tensorflow as tf
import keras_tuner as kt
from tensorflow.keras import layers, models
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------- CONFIG ----------------
DATASET_DIR = "/kaggle/input/potatoleafplantvillage/VanillaLDMAugmented"
TRAIN_DIR = os.path.join(DATASET_DIR, "train")
TEST_DIR  = os.path.join(DATASET_DIR, "test")

IMG_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 10
NUM_CLASSES = 3
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

# ---------------- LOAD FUNCTION ----------------
def load_paths_labels(directory):
    image_paths, labels = [], []
    class_names = sorted(os.listdir(directory))[:NUM_CLASSES]

    for idx, cls in enumerate(class_names):
        cls_dir = os.path.join(directory, cls)
        for f in os.listdir(cls_dir):
            image_paths.append(os.path.join(cls_dir, f))
            labels.append(idx)

    return np.array(image_paths), np.array(labels), class_names

# ---------------- LOAD DATA ----------------
X_train_all, y_train_all, class_names = load_paths_labels(TRAIN_DIR)
X_test, y_test, _ = load_paths_labels(TEST_DIR)

print("Train images:", len(X_train_all))
print("Test images (Fixed):", len(X_test))

# ---------------- DATA GENERATOR ----------------
def data_generator(paths, labels):
    while True:
        idxs = np.arange(len(paths))
        np.random.shuffle(idxs)

        for i in range(0, len(paths), BATCH_SIZE):
            batch_idx = idxs[i:i+BATCH_SIZE]
            imgs, labs = [], []

            for j in batch_idx:
                img = cv2.imread(paths[j])
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                img = img.astype("float32")
                img = preprocess_input(img)

                imgs.append(img)
                labs.append(labels[j])

            imgs = np.array(imgs)
            labs = tf.keras.utils.to_categorical(labs, NUM_CLASSES)
            yield imgs, labs

# ---------------- MODEL BUILDER (BAYESIAN) ----------------
def build_model(hp):
    base = InceptionV3(
        weights="imagenet",
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = False

    x = base.output
    x = layers.GlobalAveragePooling2D()(x)

    units = hp.Int("dense_units", 256, 1024, step=256)
    x = layers.Dense(units, activation="relu")(x)

    drop = hp.Float("dropout", 0.3, 0.6, step=0.1)
    x = layers.Dropout(drop)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(base.input, outputs)

    lr = hp.Choice("lr", [1e-3, 1e-4, 5e-5])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# ---------------- 5-FOLD CV ----------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

fold = 1
cv_scores = []

for train_idx, val_idx in skf.split(X_train_all, y_train_all):

    print(f"\n🔁 Fold {fold}")

    X_train, X_val = X_train_all[train_idx], X_train_all[val_idx]
    y_train, y_val = y_train_all[train_idx], y_train_all[val_idx]

    tuner = kt.BayesianOptimization(
        build_model,
        objective="val_accuracy",
        max_trials=5,
        directory="bayesian_tuning_inceptionv3_noaug",
        project_name=f"fold_{fold}"
    )

    tuner.search(
        data_generator(X_train, y_train),
        steps_per_epoch=len(X_train)//BATCH_SIZE,
        validation_data=data_generator(X_val, y_val),
        validation_steps=len(X_val)//BATCH_SIZE,
        epochs=EPOCHS,
        verbose=0
    )

    best_model = tuner.get_best_models(1)[0]

    val_loss, val_acc = best_model.evaluate(
        data_generator(X_val, y_val),
        steps=len(X_val)//BATCH_SIZE,
        verbose=0
    )

    print(f"Fold {fold} Validation Accuracy: {val_acc:.4f}")
    cv_scores.append(val_acc)
    fold += 1

print("\n✅ Mean CV Accuracy:", np.mean(cv_scores))

# ---------------- FINAL MODEL ----------------
final_model = tuner.get_best_models(1)[0]

# ---------------- TEST EVALUATION (FIXED) ----------------
y_true, y_pred = [], []
test_gen = data_generator(X_test, y_test)

for _ in range(len(X_test)//BATCH_SIZE):
    imgs, labs = next(test_gen)
    preds = final_model.predict(imgs, verbose=0)
    y_true.extend(np.argmax(labs, axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

test_acc = np.mean(np.array(y_true) == np.array(y_pred))
print(f"\n🎯 Final Test Accuracy (Fixed): {test_acc:.4f}")

# ---------------- REPORT ----------------
print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# ---------------- CONFUSION MATRIX ----------------
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=class_names,
            yticklabels=class_names,
            cmap="Blues")
plt.title("Confusion Matrix (Fixed Test Set)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()
